In [2]:
!pip install aix360

In [3]:
pip install sympy==1.12 --force-reinstall #fixes compatibility issues with aix360

  Using cached sympy-1.12-py3-none-any.whl.metadata (12 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
Using cached sympy-1.12-py3-none-any.whl (5.7 MB)
Using cached mpmath-1.3.0-py3-none-any.whl (536 kB)
  Attempting uninstall: mpmath
    Found existing installation: mpmath 1.3.0
    Uninstalling mpmath-1.3.0:
      Successfully uninstalled mpmath-1.3.0
  Attempting uninstall: sympy
    Found existing installation: sympy 1.12
    Uninstalling sympy-1.12:
      Successfully uninstalled sympy-1.12
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.8.0+cu126 requires sympy>=1.13.3, but you have sympy 1.12 which is incompatible.


In [112]:
import sys
import pandas as pd #loading data in table form
import numpy as np # linear algebra
import torch # import main library
import torch.nn as nn # import modules
from torch.autograd import Function # import Function to create custom activations
from torch.nn.parameter import Parameter # import Parameter to create custom activations with learnable parameters
import torch.nn.functional as F # import torch functions
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

from aix360.algorithms.die import DIExplainer

class BinarizeSTE(torch.autograd.Function):
    @staticmethod
    def forward(ctx, input):
        output = (input > 0.5).float() # sorts values to 0 or 1 based on if greater than 0.5.
        ctx.save_for_backward(input) #saving input for backward pass (best practice?)
        return output

    @staticmethod
    def backward(ctx, grad_output):
        input, = ctx.saved_tensors
        sigmoid = torch.sigmoid(10 * (input - 0.5))  # sharp sigmoid near 0.5 (imitates binarization more smoothly)
        grad_input = grad_output * sigmoid * (1 - sigmoid) * 10  # chain rule
        return grad_input

def binarize_with_ste(x):
        return BinarizeSTE.apply(x)

#for hamming distance calculations
class ThresholdBinarizeSTE(torch.autograd.Function):
    @staticmethod
    def forward(ctx, input, threshold):
        output = (input > threshold).float()
        ctx.save_for_backward(input)
        ctx.threshold = threshold
        return output

    @staticmethod
    def backward(ctx, grad_output):
        input, = ctx.saved_tensors
        threshold = ctx.threshold
        sigmoid = torch.sigmoid(10 * (input - threshold))
        grad_input = grad_output * sigmoid * (1 - sigmoid) * 10
        return grad_input, None

def threshold_binarize_with_ste(x, threshold):
      return ThresholdBinarizeSTE.apply(x, threshold)

class XorSTE(torch.autograd.Function):
    @staticmethod
    def forward(ctx, input1, input2):
      bin1 = (input1 > 0.5).float()
      bin2 = (input2 > 0.5).float()
      output = torch.logical_xor(bin1, bin2).float()
      ctx.save_for_backward(input1, input2)
      return output

    @staticmethod
    def backward(ctx, grad_output):
      input1, input2 = ctx.saved_tensors
      grad1 = grad_output * (1 - 2 * input2.int())
      grad2 = grad_output * (1 - 2 * input1.int())

      return grad1, grad2

def xor_ste(x1, x2):
      return XorSTE.apply(x1, x2)


class CoLogNet_Model(nn.Module):
  '''
  Continued Logarithm model

  continuant form, where C_n represents 2^c_n is as follows:


    C_2((C_1 * C_0) + C_0) + (C_1 * C_0)
    ------------------------------------   ...
             (C_2 * C_1) + C_1
  '''

  def __init__(self, input_size, output_size, num_ladders, depth, dropout_num, model_type, hamm_dep):

        super().__init__()
        self.input_size = input_size
        self.output_size = output_size
        self.num_ladders = num_ladders
        self.depth = depth #depth of each ladder
        self.layers = nn.ModuleList()
        self.hamm_dep = hamm_dep

        for i in range(0, output_size): #create an ensemble for each output dim
          ensemble = Ensemble(input_size, output_size, depth, num_ladders, dropout_num, model_type, hamm_dep)
          self.layers.append(ensemble)


    #computes outputs (matrix of values, later one-hot-encoded in train loop) based on inputs
  def forward(self, inputs):

        final_vector = []

        #version 1 (by ladder)
        for x in range(0, self.output_size): #for each output class
          ensemble = self.layers[x]
          ensemble_val = ensemble(inputs)
          final_vector.append(ensemble_val)

        output = torch.stack(final_vector, dim=1)
        print(f'OUTPUT {output}')
        return output  # shape: (batch_size, output_size)

class Ensemble(nn.Module):
  def __init__(self, input_size, output_size, depth, num_ladders, dropout_num, model_type, hamm_dep):
    super().__init__()
    self.input_size = input_size
    self.output_size = output_size
    self.depth = depth
    self.num_ladders = num_ladders
    self.ladders = nn.ModuleList()
    self.hamm_dep = hamm_dep
    self.model_type = model_type
    for i in range(0, num_ladders): #for each ladder in ensemble
      ladder = Ladder(input_size, depth, dropout_num, model_type, hamm_dep)
      self.ladders.append(ladder) #add new ladder to ladders

    if (model_type == 'CoLogNetB-H'):
      self.linear_layer = nn.Linear(num_ladders, output_size)

  def forward(self, inputs): #maybe not actually forward method... returns sum of ladder outputs, to be used as one of output dims

    ladder_vals = []
    for ladder in self.ladders:
      ladder_val = ladder(inputs)
      ladder_vals.append(ladder_val)

    stacked = torch.stack(ladder_vals, dim=0)  # (num_ladders, batch_size), combining into one big tensor

    if (self.model_type == 'CoLogNetB-H'):
      output = self.linear_layer(stacked)
    else:
      output = stacked.mean(dim=0)  # (batch_size), averaging across each column
    return output

class Ladder(nn.Module):
  def __init__(self, input_size, depth, dropout_num, model_type, hamm_dep):
    super().__init__()
    self.input_size = input_size
    self.depth = depth
    self.rung_size = 64 * hamm_dep

    #self.bias = nn.Parameter(torch.Tensor(depth))

    if (model_type == 'CoLogNetB-H'): #version 2, where everything (inputs, weights) is binarized and use hamming distance (STILL IN PROGRESS)
      '''
      self.weight = nn.Parameter(torch.Tensor(depth * self.rung_size, input_size))
      self.hamming_weight = nn.Parameter(torch.Tensor(depth * self.rung_size, input_size))
      '''
      self.reset_parameters_binary()

    else:
      self.weight = nn.Parameter(torch.Tensor(depth, input_size)) #depth rows & input_size columns
      self.reset_parameters()

    self.dropout = nn.Dropout(p=dropout_num) # optional dropout
    self.model_type = model_type #whether to do as cofr, colog, or cologB

  def reset_parameters(self):
    nn.init.xavier_uniform_(self.weight)

  def reset_parameters_binary(self):
    self.reshape_layer = nn.Linear(self.input_size, self.rung_size)

    # tensor of probabilities - 50% chance of 0 or 1
    probabilities = torch.full((self.depth, self.rung_size), 0.5)

    # binary tensor from the probabilities
    binary_tensor = torch.bernoulli(probabilities)

    self.hamming_layer = nn.Parameter(binary_tensor)


  def forward(self, inputs): #returns vector [b_0, b_1, ..., b_n] where b_i is a rung of the ladder
    #return torch.matmul(self.weight, inputs)

    # inputs: (batch_size, input_size)
    # weight: (depth, input_size)


    if (self.model_type == 'CoLogNetB-H'):

      reshaped_inputs = self.reshape_layer(inputs) #reshape inputs to dims (rung-size)
      inputs_bin = binarize_with_ste(reshaped_inputs) #binarize
      #print(f'INPUTS: **  {inputs_bin.shape} **')

      bs = inputs.shape[0]
      repeated_inputs = inputs_bin.repeat(self.depth, 1, 1) #(depth, bs, rung_size)
      repeated_inputs = torch.transpose(repeated_inputs, 0, 1) # (bs, depth, rung_size)
      repeated_weights = self.hamming_layer.repeat(bs, 1, 1) #(bs, depth, rung_size)

      distances = xor_ste(repeated_inputs, repeated_weights)
      #print(f' XOR: ** {distances.shape} ** ')

      '''
      for rung in self.depth:
        xor_vals = xor_ste(inputs_bin, repeated_weights) #DON'T WANT SAME WEIGHTS!!
        popcount = xor_vals.sum(dim=1)
      '''

      #torch.vmap(lambda batch: xor_ste(batch, self.hamming_layer))(inputs_bin)

      #Hamming distance between inputs & weights (both binary):
      #distances = xor_ste(inputs_bin, self.hamming_layer) #apply xor

      # Reshape the tensor into a 2D tensor with rung_size columns
      '''
      bs = distances.shape[0]
      grouped_by_rung = distances.reshape(bs, self.depth, self.rung_size)

      # Sums every rung to get hamming distance for each rung
      summed_rungs = torch.sum(grouped_by_rung, dim=-1) #--- ONLY WANT TO DO THIS OVER EACH ROW
      '''
      summed_rungs = distances.sum(dim=2) #summed per rung

      #print(f'POPCOUNT: ** {summed_rungs.shape} **')

      threshold = self.rung_size / 2

      #binarize hamming distances based on threshold
      #output = torch.where(summed_rungs > threshold, 1, 0)
      output = threshold_binarize_with_ste(summed_rungs, threshold)

      #print(f'THRESHOLD: ** {output.shape} **')

      output = torch.transpose(output, 0, 1) #flips rows and columns
      #already depth x batch_size though?

    else:
      output = F.linear(inputs, self.weight)  # output: (r = batch_size, c = depth)

      output = torch.transpose(output, 0, 1) #flips rows and columns

      if (self.model_type == 'CoLogNetB'):
        output = binarize_with_ste(output) #for CoLogNetB
      else:
        output = torch.sigmoid(output) #for CoFrNet and CoLogNet

    output = self.recurrence_formula(output, self.model_type) #apply recurrence function. should be vector of len batch size

    output = self.dropout(output)
    print(f' RECURRENCE: ** {output} ** ')
    return output


  @staticmethod
  def recurrence_formula(C: torch.Tensor, model_type): #also works for 2d matrices because formula is done to columns in parallel

    """
    Compute x_n = An / Bn using:

    C_0
    ---,
     1

    C_1 * C_0 + C_0
    ---------------,
          C_1

      C_2((C_1 * C_0) + C_0) + (C_1 * C_0)
    ------------------------------------   ...
              (C_2 * C_1) + C_1

    A_0 = C_0
    A_1 = C_1 * C_0 + C_0
    A_n = C_n * A_{n-1} + A_{n-2}

    B_0 = 1
    B_1 = C_1
    B_n = C_n * B_{n-1} + C_{n-1}

    Takes in 1d tensor of C_i values,
    Returns value of A_n/B_n
    """
    # clamp to prevent A_n or B_n exploding
    #c = torch.clamp(c, min=-3.0, max=3.0)  # tighter than before

    n = len(C)

    if n == 1: # base case
      return C[0]

    if (model_type == 'CoFrNet'):
      A_prev = C[0]
      A_curr = C[1] * C[0] + 1
      B_prev = 1.0
      B_curr = C[1]

      for i in range(2, n):
          A_next = C[i] * A_curr + A_prev
          B_next = C[i] * B_curr + 1
          A_prev, A_curr = A_curr, A_next
          B_prev, B_curr = B_curr, B_next

    else: #CoLogNet or CoLogNetB
      C = torch.pow(2, C)

      A_prev = C[0]
      A_curr = C[0] * (C[1] + 1)
      B_prev = 1.0
      B_curr = C[1]

      for i in range(2, n):
          A_next = C[i] * A_curr + A_prev
          B_next = C[i] * B_curr + C[i-1]
          A_prev, A_curr = A_curr, A_next
          B_prev, B_curr = B_curr, B_next

    return A_curr / B_curr

    '''
    n = len(C)

    if n == 1:
      return torch.pow(2, C[0])

    A_0 = [C[0]]
    B_0 = [1]
    A_1 = [C[0] + C[1], C[0]]
    B_1 = [C[1]]
    A_n = Recurrence_Term(A_1, A_0)
    B_n = Recurrence_Term(B_1, B_0)

    for i in range(2, n):
        A_n.advance_a(C[i])
        B_n.advance_b(C[i], C[i-1])

    return A_n.eval() / B_n.eval()
    '''


class Recurrence_Term():
  #keeps track of list of 2^x terms in recurrence formula
  def __init__(self, terms, prev):
    self.terms = terms #current terms
    self.prev = prev #previous terms (used for A_n)

  def advance_a(self, c_n): # A_n = C_n * A_{n-1} + A_{n-2}
    curr = self.terms #saving current to be used as prev in next round
    self.mult(c_n)
    self.add(self.prev)
    self.prev = curr

  def advance_b(self, c_n, c_prev): #B_n = C_n * B_{n-1} + C_{n-1}
    self.mult(c_n)
    self.add([c_n-1])

  def mult(self, num): #add power (0 or 1) to each term (represents multiplying by 2^0 or 2^1)
    self.terms = list(map(lambda x: x + num, self.terms))

  def add(self, new_terms): #adds new terms to list of terms
    self.terms = self.terms + new_terms

  def eval(self): #evaluate the recurrence term to it's final value
    sum = 0
    for i in range(len(self.terms)):
      curr = self.terms[i]
      val = torch.pow(2, curr)
      sum += val
    return sum

class CoLogNet_Explainer(DIExplainer):
    def __init__(self, colognet_model):
        self.model = colognet_model
        self.input_size = colognet_model.input_size
        self.output_size = colognet_model.output_size
        self.weight = nn.Parameter(torch.Tensor(self.output_size, self.input_size))

    def set_params(self, *argv, **kwargs):
        """
        Set parameters for the explainer.
        """
        pass

    def get_accuracy(self, xtest, ytest):

        #results = self.model(xtest).detach().numpy()
        with torch.no_grad():
          results = self.model(xtest).cpu().numpy()

        idx = np.argmax(results, axis = -1)
        results = np.zeros(results.shape)
        results[ np.arange(results.shape[0]), idx] = 1
        results = results.argmax(axis = -1)

        numTotal = 0
        numCorrect = 0
        for i in range(0, len(results)):
            if results[i] == ytest[i]:
                numCorrect = numCorrect + 1
            numTotal = numTotal + 1
        #print("Accuracy: ", numCorrect/numTotal)
        accuracy = float(numCorrect/numTotal)
        return accuracy


    def explain(self, explain_mode, max_layer_num = 10, var_num = 6):
        '''
        Provides Explanations of CoFrNet Model

        Args:
        explain_mode: either "importances" or "print_co_fr", will raise exception if not one of these two options
        max_layer_num: For "print_co_fr": Choose Depth of Ladder to Show, Default 10
        var_num: For "print_co_fr": Variable (index of input feature) for Which to Display Ladder, Default 6
        '''

        def importances():
            #final_layer_weights = vars(self.model.layers[-1])['_parameters']['weight'].data.numpy()
            final_layer_weights = self.model.layers[-1].ladders[0].weight.data.numpy()
            weights_by_node = final_layer_weights.T
            averaged = np.average(weights_by_node, axis = 1)
            copy_averaged = averaged.copy()
            print(copy_averaged)
            num_important_to_print = 3
            for x in range(0, num_important_to_print):
                min_idx = np.argmax(copy_averaged)
                print("The number " + str(x+1) + " most important input feature was the " + str(min_idx+1) + "th one.")
                copy_averaged[np.argmax(copy_averaged)] = copy_averaged[np.argmin(copy_averaged)]
                #print(vars(self.model.layers[-1])['_parameters']['weight'].data.numpy().T)


        def print_co_fr(max_layer_num = 10, var_num = 6):
            #max_layer_num = chosen depth of ladder to show (10 layers, index would be 9)
            #var_num = variable for which to display ladder
            thingToPrint = ""
            for layerNum in range(0, max_layer_num-1):
                temp = vars(self.model.layers[layerNum])
                print()
                print("LayerNum: ", layerNum)
                val = (temp['_parameters']['weight'].data[var_num][var_num]).numpy()
                print("Val: ", val)
                bias = temp['_parameters']['bias'].data[var_num].numpy()
                print("Bias: ", bias)
                if (bias > (.01 * val)):
                    print(str(bias))
                    combined = "("+str(val) + "*x + " + str(bias)+")"
                    print("Combined: ", combined)
                else:
                    print("hi")
                    combined = "(" + str(val)+"*x" + "+0)"
                    print("Combined: ", combined)
                print()
                thingToPrint = "1/(" + combined + " + (" + thingToPrint + "))"

            print(thingToPrint)
            return thingToPrint

        if explain_mode == "importances":
            importances()
        elif explain_mode == "print_co_fr":
            print_co_fr(max_layer_num, var_num)
        else:
            raise Exception("explain_mode must be either 'importances' or 'print_co_fr'")


In [5]:
class MLP(nn.Module):

  def __init__(self, input_size, output_size, num_layers):
    super().__init__()
    self.input_size = input_size
    self.output_size = output_size
    self.num_layers = num_layers # not including activation layers

    self.init_layers()

  def init_layers(): #initialize layers based on num_layers
    hidden_layer = nn.Linear(input_size, input_size)
    activation_layer = nn.ReLU()
    output_layer = nn.Linear(input_size, output_size)

    layers = []

    for i in range(self.num_layers - 1):
      layers.append(hidden_layer)
      layers.append(activation_layer)

    layers.append(output_layer)

    self.linear_relu_stack = nn.Sequential(*layers)

  # forward pass: perform operations (linear/nonlinear) on input data to get prediction
    def forward(self, x):
        logits = self.linear_relu_stack(x)
        return logits

In [6]:
#TENSORBOARD (also scattered throughout)
'''
!pip install tensorboard

from torch.utils.tensorboard import SummaryWriter
from datetime import datetime

#writer = SummaryWriter('co_log_net')
writer = SummaryWriter(f'co_log_net/{datetime.now().strftime("%Y%m%d-%H%M%S")}')
'''

'\n!pip install tensorboard\n\nfrom torch.utils.tensorboard import SummaryWriter\nfrom datetime import datetime\n\n#writer = SummaryWriter(\'co_log_net\')\nwriter = SummaryWriter(f\'co_log_net/{datetime.now().strftime("%Y%m%d-%H%M%S")}\')\n'

In [3]:
#UTILS

'''
author: @ishapuri101
'''

import numpy as np # linear algebra
from sklearn.preprocessing import MinMaxScaler
from tqdm import tqdm
import pandas as pd #loading data in table form
from torch.utils.data import Dataset
import torch # import main library
import torch.nn as nn # import modules
from torch.autograd import Function # import Function to create custom activations
from torch.nn.parameter import Parameter # import Parameter to create custom activations with learnable parameters
import torch.nn.functional as F # import torch functions
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
import torch.optim as optim
#import random
from torch.utils.data import DataLoader

import matplotlib.pyplot as plt


def process_data(first_column_csv, last_column_csv, web_link = None, data_filename = None):
    #data_filename: filename of data source
    #first_column_csv: index (starting from 0) of first column to include in dataset
    #last_column_csv: index (starting from 0) of last column to include in dataset. Use -1 if you want to include all of the columns.

    import pandas as pd

    if web_link is not None:
        pathname = web_link
    else:
        pathname = data_filename #'datasets/' + data_filename
    df=pd.read_csv(pathname, sep=',',header=0, lineterminator='\r')
    if last_column_csv != -1:
        last_column_csv = last_column_csv + 1
    df = df.sample(frac = 1)
    X = df.iloc[:, first_column_csv : last_column_csv].values

    y = df.iloc[:,-1].values.T




    from sklearn.preprocessing import LabelEncoder
    le = LabelEncoder()
    y = le.fit_transform(y)

    from sklearn.preprocessing import StandardScaler
    sc = MinMaxScaler(feature_range=(0,1))
    X = sc.fit_transform(X)
    X.argmax()

    seeds = [1, 10, 100, 555, 9897]
    seed = seeds[2]

    from sklearn.model_selection import train_test_split
    X_train, X_test, y_train, y_test = train_test_split(X,
                                                        y,
                                                        test_size = 0.3,
                                                        random_state = seed,
                                                        shuffle = True)
    X_train, X_val, y_train, y_val = train_test_split(X_train,
                                                        y_train,
                                                        test_size=0.05,
                                                        random_state=seed,
                                                        shuffle = True)

    #CONVERTING TO TENSOR
    tensor_x_train = torch.Tensor(X_train)
    tensor_x_val = torch.Tensor(X_val)
    tensor_x_test = torch.Tensor(X_test)

    tensor_y_val = torch.Tensor(y_val).long()
    tensor_y_train = torch.Tensor(y_train).long()
    tensor_y_test = torch.Tensor(y_test).long()

    return tensor_x_train, tensor_y_train, tensor_x_val, tensor_y_val, tensor_x_test, y_test


def onehot_encoding(label, n_classes):
    """Conduct one-hot encoding on a label vector."""
    label = label.view(-1)
    onehot = torch.zeros(label.size(0), n_classes).float().to(label.device)
    onehot.scatter_(1, label.view(-1, 1), 1)

    return onehot


def train(model, dataloader, num_classes, lr = 0.01, momentum = 0.9, epochs = 20): #lr changed from 0.001 to improve accuracy
    criterion = nn.CrossEntropyLoss()
    #criterion = nn.MSELoss(reduction="sum")
    #optimizer = optim.SGD(model.parameters(), lr=lr, momentum=momentum)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4) #don't use weight decay, it decreases accuracy

    global_step = 0

    EPOCHS = epochs

    best_loss = float('inf')
    patience = 5  # Number of epochs to wait for improvement
    counter = 0

    for epoch in range(EPOCHS):  # loop over the dataset multiple times
        #print("Epoch: ", epoch)#
        running_loss = 0.0
        #for i, data in enumerate(trainloader, 0):
        for i, batch in tqdm(enumerate(dataloader)):

            # get the inputs; data is a list of [inputs, labels]
            # forward + backward + optimize

            batch['tabular'].requires_grad=True

            outputs = model(batch['tabular'])

            one_hot_encoded_target = onehot_encoding(batch['target'], num_classes)

            #loss = criterion(outputs, batch['target'])
            loss = criterion(outputs, one_hot_encoded_target)

            #writer.add_scalar('Training Loss', loss.item(), epoch)
            # Log every batch, not just every epoch
            #writer.add_scalar('Training Loss', loss.item(), global_step)
            global_step += 1

            # zero the parameter gradients
            optimizer.zero_grad()

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0) #gradient clipping to improve accuracy
            optimizer.step()

            # print statistics
            running_loss += loss.item()
            #print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 2000:.3f}')

        if running_loss < best_loss:
          best_loss = running_loss
          counter = 0
          # Save model checkpoint
          torch.save(model.state_dict(), 'best_model.pth')
        else:
          counter += 1
          if counter >= patience:
            print(f"Early stopping at epoch {epoch} as validation loss did not improve for {patience} epochs.")
            return running_loss
        '''
        for name, param in model.named_parameters():
          if param.requires_grad:
              writer.add_histogram(name, param.data, epoch)

        writer.close()
        '''

        #print("Loss:", running_loss)#temporarily commented out#
    return running_loss
    #print('Finished Training')#

def simple_binarize(inputs):
  mean = inputs.mean()
  inputs = inputs - mean #center inputs around avg
  inputs = torch.sin(inputs) #apply sin --> values between -1 and 1
  inputs = torch.where(inputs > 0, 1.0, 0.0) # if pos, 1, if neg, 0

class OnlyTabularDataset(Dataset):
    def __init__(self, values, label):
        self.values = values
        self.label = label

    def __len__(self):
        return len(self.label)

    def __getitem__(self, index):
        return {
            'tabular': self.values[index].clone().detach().float(),
            'target' :  self.label[index].clone().detach().long()
            }

In [4]:
#TEST

import unittest
import os
import shutil
import sys

from torch.utils.data import DataLoader
from tqdm import tqdm
import numpy as np
from torch.utils.data import Dataset
import torch # import main library
import torch.nn as nn # import modules
from torch.autograd import Function # import Function to create custom activations
from torch.nn.parameter import Parameter # import Parameter to create custom activations with learnable parameters
import torch.nn.functional as F # import torch functions
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
import torch.optim as optim
from sklearn.datasets import load_breast_cancer, load_wine, load_linnerud, load_diabetes, load_iris, load_digits
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

import random
import pandas as pd
import matplotlib.pyplot as plt

#WAVEFORM: *
input_size = 40
output_size = 3

first_column_csv = 0
last_column_csv = -1


web_link = 'http://www.dropbox.com/s/qtdv1teptf097zl/waveformnoise.csv?dl=1'
#*

'''
#MAGIC
web_link = 'https://www.dropbox.com/s/yourmagicfile.csv?dl=1'

input_size = 10
output_size = 2

first_column_csv = 0
last_column_csv = -1
'''

'''
#CREDIT CARD:

'''

tensor_x_train, tensor_y_train, tensor_x_val, tensor_y_val, tensor_x_test, y_test = process_data(first_column_csv = first_column_csv,
                                                                                                        last_column_csv = last_column_csv,
                                                                                                        web_link=web_link)
train_dataset = OnlyTabularDataset(tensor_x_train,
                                        tensor_y_train)

def compute_mlp_layers(input_size, output_size, ladders, depth): #compute # of layers mlp should have to have same # parameters as colognet
# (params_MLP = (num_layers - 1) x (input_size^2 + input_size) + (input_size x output_size + output_size)
    numerator = (input_size * output_size * ladders * depth) - (input_size * output_size + output_size)
    denominator = input_size**2 + input_size
    num_layers = 1 + numerator / denominator
    return max(1, round(num_layers))  # Ensure at least 1 layer

def train_test_loop(depth, num_ladders, learning_rate, batch_size, dropout_num, model_type, hamm_dep):

  dataloader = DataLoader(train_dataset, batch_size)

  if (model_type == "MLP"):
    model = MLP(input_size, output_size, compute_mlp_layers(input_size, output_size, num_ladders, depth))
  else:
    model = CoLogNet_Model(input_size, output_size, num_ladders, depth, dropout_num, model_type, hamm_dep)

  train(model, dataloader, output_size, learning_rate)

  total_params = sum(p.numel() for p in model.parameters() if p.requires_grad) #optional: print total parameters of model
  #print("Total trainable parameters:", total_params)#

  '''
  writer.flush()#
  writer.close()#
  '''


  explainer = CoLogNet_Explainer(model)
  accuracy = explainer.get_accuracy(tensor_x_test, y_test)

  #print("Accuracy:", accuracy)#
  return accuracy, total_params



In [113]:
train_test_loop(2, 1, 0.01, 1, 0.00, "CoLogNetB-H", 1)

0it [00:00, ?it/s]

 RECURRENCE: ** tensor([2.], grad_fn=<DivBackward0>) ** 
 RECURRENCE: ** tensor([4.], grad_fn=<DivBackward0>) ** 
 RECURRENCE: ** tensor([4.], grad_fn=<DivBackward0>) ** 
OUTPUT tensor([[[-0.7225, -1.7477, -1.7013],
         [ 1.2316,  2.1933,  3.9124],
         [-1.9136,  0.0046, -2.2751]]], grad_fn=<StackBackward0>)


RuntimeError: expected scalar type Long but found Float

In [ ]:
# === Mount Google Drive ===
from google.colab import drive
drive.mount('/content/drive')

# === Imports ===
import os
import shutil
import pandas as pd

# === File Path ===
drive_file_path = '/content/drive/MyDrive/CoFrNet_CoLogNet_Data.h5'
backup_file_path = '/content/drive/MyDrive/backup_CoFrNet_CoLogNet_Data.h5'

# === Create Empty HDF5 File if not already initialized ===
if not os.path.exists(drive_file_path):
    columns = ['Dataset', 'Depth', 'Ladders', 'Learning rate', 'Total parameters', 'Variant', 'Accuracy']
    df_empty = pd.DataFrame(columns=columns)
    df_empty.to_hdf(
        drive_file_path,
        key='tests_dataframe',
        mode='w',
        format='table',
        data_columns=True
    )
    print("HDF5 file created.")
else:
    print("]File already exists — will append results.")


In [ ]:
import pandas as pd

#TESTING LOOP
def test_loop (d_vals, l_vals, lr_vals, bs, acc_trials, model_types): #inc = amt to increment by
  total_rounds = len(d_vals) * len(l_vals) * len(lr_vals) * len(model_types) * acc_trials
  curr_round = 1

  # Load existing trials from file to skip duplicates
  try:
      df_existing = pd.read_hdf(drive_file_path, 'tests_dataframe')
      completed = set(zip( #combines into one tuple
          df_existing['Depth'],
          df_existing['Ladders'],
          df_existing['Learning rate'],
          df_existing['Variant'],
          df_existing.groupby(['Depth', 'Ladders', 'Learning rate', 'Variant']).cumcount() #assigns num to each trial
      ))
      print(f"Found {len(completed)} completed trials. Resuming from last point.")
  except Exception as e:
      completed = set()
      print(f"Could not load existing trials (starting fresh): {e}")

  for d in d_vals:
    for l in l_vals:
      for lr in lr_vals:
        for variant in model_types:
          for trial in range(acc_trials):
            try:
              # Create a unique ID for each trial
              id_tuple = (d, l, lr, variant, trial)

              # Skip this trial if already completed
              if id_tuple in completed:
                  print(f"Skipping already completed trial {id_tuple}")
                  curr_round += 1
                  continue

              accuracy, params = train_test_loop(
                  depth=d,
                  num_ladders=l,
                  learning_rate=lr,
                  batch_size=bs,
                  dropout_num=0.0,
                  model_type=variant
              )

              row = ['Waveform', d, l, lr, params, variant, accuracy]
              new_row_df = pd.DataFrame([row], columns=[
                  'Dataset', 'Depth', 'Ladders', 'Learning rate',
                  'Total parameters', 'Variant', 'Accuracy'
              ])

              with pd.HDFStore(drive_file_path, mode='a') as store:
                  store.append(
                      'tests_dataframe',
                      new_row_df,
                      format='table',
                      data_columns=True,
                      min_itemsize={'Dataset': 10, 'Variant': 10}
                  )

              print(f'Trial {curr_round} / {total_rounds}')
              print(row)

              # BACKUP every 10 trials
              if curr_round % 10 == 0:
                  backup_path = '/content/drive/MyDrive/backup_CoFrNet_CoLogNet_Data.h5'
                  shutil.copy(drive_file_path, backup_path)
                  print(f'Backup created at trial {curr_round}')

            except Exception as e:
                print(f"Trial {curr_round} failed: {e}")

            curr_round += 1

d_vals = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
l_vals = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80]
lr_vals = [0.001, 0.01] #currently skipping 0.5, 0.1 to be faster
model_types = ['CoLogNetB']
test_loop(d_vals=d_vals, l_vals=l_vals, lr_vals=lr_vals, bs=100, acc_trials=5, model_types=model_types)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import seaborn as sns

drive_file_path = '/content/drive/MyDrive/CoFrNet_CoLogNet_Data.h5'

# Check if file exists
if os.path.exists(drive_file_path):
    # Load the data
    try:
        df = pd.read_hdf(drive_file_path, key='tests_dataframe')
        print("Loaded HDF5 file successfully.")

        # Average accuracy over trials
        df_avg = df.groupby(['Depth', 'Ladders','Learning rate', 'Variant']).agg({
            'Accuracy': 'mean',
            'Total parameters': 'first'  # Assuming total parameters is same across trials
})
        '''
        # Take max accuracy for each parameter number
        df_max = df_avg.groupby(['Total parameters', 'Variant']).apply(
            lambda g: g.loc[g['Accuracy'].idxmax()]
        )
        '''
        # Get index of the row with max accuracy for each (Total parameters, Variant)
        idx = df_avg.groupby(['Total parameters', 'Variant'])['Accuracy'].idxmax()

        # Select those rows
        df_max = df_avg.loc[idx].copy().reset_index()

        print(df_avg.columns)
        print(df_avg.index.names)
        print(df_avg.reset_index().head())

        #sanity check: what are max accuracies overall for each model
        df_max_sample = df_max.groupby(['Variant']).apply(
            lambda g: g['Accuracy'].max()
        )
        print(df_max_sample)

        #Plot line plot
        sns.set(style='whitegrid', context='notebook')
        plt.figure(figsize=(8, 6)) # Optional: set figure size
        sns.lineplot(x='Total parameters', y='Accuracy', hue='Variant', data=df_max, ci=None)
        plt.title('Accuracy over Total Parameters by Model Variant')
        plt.xlabel('Total parameters')
        plt.ylabel('Accuracy')
        plt.legend(title='Variant') # Optional: customize legend title
        plt.show()

        #Heatmaps
        df_avg = df_avg.reset_index()

        df_cofr = df_avg[df_avg['Variant'] == 'CoFrNet']
        df_colog = df_avg[df_avg['Variant'] == 'CoLogNet']
        df_cologb = df_avg[df_avg['Variant'] == 'CoLogNetB']
        cofr_hm = df_cofr.pivot_table(index='Depth', columns='Ladders', values='Accuracy', aggfunc='max')
        colog_hm = df_colog.pivot_table(index='Depth', columns='Ladders', values='Accuracy', aggfunc='max')
        cologb_hm = df_cologb.pivot_table(index='Depth', columns='Ladders', values='Accuracy', aggfunc='max')

        # Plot heatmaps
        plt.figure(figsize=(12, 4))

        plt.subplot(1, 3, 1)
        sns.heatmap(cofr_hm, annot=False, fmt=".2f", cmap="YlGnBu")
        plt.title('CoFrNet')

        plt.subplot(1, 3, 2)
        sns.heatmap(colog_hm, annot=False, fmt=".2f", cmap="YlGnBu")
        plt.title('CoLogNet')

        plt.subplot(1, 3, 3)
        sns.heatmap(cologb_hm, annot=False, fmt=".2f", cmap="YlGnBu")
        plt.title('CoLogNetB')

        plt.tight_layout()
        plt.show()

    except Exception as e:
        print("Error loading HDF5 file:", e)
else:
    print("File does not exist at:", drive_file_path)

In [ ]:
'''
#HEATMAP
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_csv("heatmap_data.csv")
df.columns = ['depth', 'ladders', 'lr', 'total params', 'CoFrNet', 'CoLogNet', 'CoLogNetB']

pivot_df = df.pivot_table(index='depth', columns='ladders', values='CoLogNetB', aggfunc='mean')

sns.heatmap(pivot_df)

plt.show()
'''

In [ ]:
#DATA COLLECTION FOR HYPERPARAMETERS (MANUAL)
'''
import random

data = []


def train_data_collect(depth, ladders, lr, bs, dp):
  accuracy, params = train_test_loop(depth, ladders, lr, bs, dp)
  for i in range(3): #might need to increase for consistency but takes longer
    curr_accuracy, params = train_test_loop(depth, ladders, lr, bs, dp)
    accuracy = (accuracy + curr_accuracy) / 2

  specs = f'dep = {depth}, lad = {ladders}, lr = {lr}, batch = {bs}, drop = {dp}' #temporary until figure out array format
  results = f'Accuracy: {accuracy} .... Params: {params}'
  cell = [specs, results]
  return cell, accuracy

def rand_tune(num_trials):
  best_acc = 0
  best_acc_cell = []
  for i in range(num_trials):
    print(f'Trial {i}/{num_trials}')
    depth = random.randint(2, 15)
    ladders = random.randint(2, 15)
    lr = random.uniform(1e-5, 1e-1)
    bs = random.randint(1, 5) * 100
    dp = 0 #random.randint(0, 5) * 0.1
    cell, accuracy = train_data_collect(depth, ladders, lr, bs, dp)
    if (accuracy > best_acc):
      best_acc = accuracy
      best_acc_cell = cell
    #data.append(cell)

  print(best_acc)
  print(best_acc_cell)

def print_2d(array):
  for row in array:
    for element in row:
        print(element, end=" ") # Print each element with a space
    print() # Move to the next line after printing all elements in a row

rand_tune(10)

#print_2d(data)
'''

In [ ]:
'''
!pip install -U ray[tune]
'''

In [ ]:
#HYPERPARAMETER TUNING
'''
from ray import tune
from ray.tune.schedulers import ASHAScheduler

from ray.tune import with_parameters

import time

# Define your training function that accepts a 'config' dictionary
def train_model(config):
    print("🚀 Trial starting with config:", config)
    time.sleep(1)  # For visibility

    batch_size = config["batch_size"]
    num_ladders = int(config["num_ladders"])  # cast to int if using tune.uniform
    network_depth = int(config["network_depth"])  # cast to int
    dropout_num = config["dropout_num"]
    lr = config["lr"]

    input_size = 40
    output_size = 3

    first_column_csv = 0
    last_column_csv = -1


    web_link = 'http://www.dropbox.com/s/qtdv1teptf097zl/waveformnoise.csv?dl=1'
    tensor_x_train, tensor_y_train, tensor_x_val, tensor_y_val, tensor_x_test, y_test = process_data(first_column_csv = first_column_csv,
                                                                                                            last_column_csv = last_column_csv,
                                                                                                            web_link=web_link)
    train_dataset = OnlyTabularDataset(tensor_x_train,
                                            tensor_y_train)

    dataloader = DataLoader(train_dataset, batch_size, num_workers=0)

    model = CoLogNet_Model(input_size, output_size, num_ladders, network_depth, dropout_num)

    validation_loss = train(model, dataloader, output_size, lr)
    explainer = CoLogNet_Explainer(model)
    validation_accuracy = explainer.get_accuracy(tensor_x_test, y_test)

    tune.report(loss=validation_loss, accuracy=validation_accuracy)

# Define the search space for hyperparameters
config = {
    "batch_size": tune.choice([32, 64, 128, 256, 512]),
    "num_ladders": tune.randint(2, 26),
    "network_depth": tune.randint(2, 26),
    "dropout_num": tune.uniform(0.0, 0.5),
    "lr": tune.loguniform(1e-8, 1e-1)
}

# Run the tuning experiment
analysis = tune.run(
    train_model,
    config=config,
    num_samples=10, # Number of different hyperparameter combinations to try
    scheduler=ASHAScheduler(metric="loss", mode="min"),
    resources_per_trial={"cpu": 0.5},     # Necessary to launch trials
    raise_on_failed_trial=False,
    fail_fast=False,
    verbose=1,
)
'''

In [ ]:

'''
%load_ext tensorboard
%tensorboard --logdir co_log_net # specified log directory
'''